<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/triton_Kimi_topo_cbp_gguf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## setup

In [ ]:
!pip install -U bitsandbytes>=0.46.1 -q

In [ ]:
!pip install -U fla-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 18.8 MB/s  0:00:00


In [ ]:
!pip install compressed-tensors>=0.15.0

In [ ]:
!nvidia-smi

Sun Nov 22 22:52:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls -ltha /content/drive/MyDrive/GGUF_Models/

In [ ]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu130 \
  --no-cache-dir

In [ ]:
!pip install -q "transformers==4.51.3" "accelerate" "bitsandbytes>=0.45"

## train

In [1]:
import os
import warnings

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
warnings.filterwarnings("ignore", category=FutureWarning, message=r".*rope_scaling.*")

import gc
import math
import hashlib
import random
import torch
from torch.optim import Optimizer
from torch.utils.data import DataLoader
import transformers.utils.generic as _tf_generic
import transformers.utils.import_utils as _tf_import_utils
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()

# Compatibility shims for Kimi remote code on newer transformers releases
# (harmless if you pin transformers==4.51.3 instead)
if not hasattr(_tf_generic, "OutputRecorder"):

    class _OutputRecorder:
        def __init__(self, *args, **kwargs):
            self.args = args
            self.kwargs = kwargs
            self.outputs = {}

        def record(self, key, value):
            self.outputs[key] = value

        def __call__(self, *args, **kwargs):
            return self

    _tf_generic.OutputRecorder = _OutputRecorder

if not hasattr(_tf_import_utils, "is_torch_fx_available"):

    def _is_torch_fx_available() -> bool:
        return True

    _tf_import_utils.is_torch_fx_available = _is_torch_fx_available

from transformers import (
    AutoConfig,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    get_cosine_schedule_with_warmup,
)
from transformers.dynamic_module_utils import get_class_from_dynamic_module
from datasets import load_dataset

try:
    import triton
    import triton.language as tl

    HAS_TRITON = True
except ImportError:
    triton = None
    tl = None
    HAS_TRITON = False

# =====================================================================
# 0. SEED & DEVICE
# =====================================================================
SEED = 123
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
use_cuda = num_gpus > 0
if not use_cuda:
    raise RuntimeError("4-bit bitsandbytes loading requires a CUDA GPU.")
primary_device = torch.device("cuda:0")

# =====================================================================
# 1. OPTIMIZER: AdamW with bf16 states, stochastic rounding and
#    error-compensated bf16 parameter updates (fused Triton kernel +
#    PyTorch fallback). Optional "anchor rows" for an embedding that is
#    actually being trained.
# =====================================================================
PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]


def _stochastic_round_(src_fp32: torch.Tensor, dst_bf16: torch.Tensor) -> None:
    """Round fp32 -> bf16 stochastically. NOTE: src_fp32 is modified in place."""
    bits = src_fp32.view(torch.int32)
    bits.add_(torch.randint_like(bits, 0, 1 << 16))
    bits.bitwise_and_(-65536)
    dst_bf16.copy_(src_fp32)


if HAS_TRITON:

    @triton.jit
    def _topo_adamw_fused_step_kernel(
        p_ptr,
        grad_ptr,
        m_buf_ptr,
        v_buf_ptr,
        comp_ptr,
        n_elements,
        lr,
        beta1,
        beta2,
        eps,
        wd,
        bias_corr1,
        bias_corr2,
        seed_m,
        seed_v,
        P_IS_BF16: tl.constexpr,
        BLOCK_SIZE: tl.constexpr,
    ):
        pid = tl.program_id(axis=0)
        offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
        mask = offsets < n_elements

        p_raw = tl.load(p_ptr + offsets, mask=mask, other=0.0)
        g = tl.load(grad_ptr + offsets, mask=mask, other=0.0).to(tl.float32)
        m_prev = tl.load(m_buf_ptr + offsets, mask=mask, other=0.0).to(tl.float32)
        v_prev = tl.load(v_buf_ptr + offsets, mask=mask, other=0.0).to(tl.float32)
        comp_prev = tl.load(comp_ptr + offsets, mask=mask, other=0.0).to(tl.float32)

        w = p_raw.to(tl.float32)

        m = beta1 * m_prev + (1.0 - beta1) * g
        v = beta2 * v_prev + (1.0 - beta2) * g * g

        m_hat = m / bias_corr1
        v_hat = v / bias_corr2
        denom = tl.sqrt(v_hat) + eps
        step_update = -lr * (m_hat / denom + wd * w) + comp_prev

        total = w + step_update
        if P_IS_BF16:
            new_p = total.to(tl.bfloat16)
            new_comp = total - new_p.to(tl.float32)
            tl.store(p_ptr + offsets, new_p, mask=mask)
            tl.store(comp_ptr + offsets, new_comp.to(tl.bfloat16), mask=mask)
        else:
            tl.store(p_ptr + offsets, total, mask=mask)
            tl.store(comp_ptr + offsets, tl.zeros_like(total).to(tl.bfloat16), mask=mask)

        # Stochastic rounding of the moments to bf16. Jitter is cast to int32
        # explicitly because tl.randint returns uint32.
        m_bits = m.to(tl.int32, bitcast=True)
        m_jitter = (tl.randint(seed_m, offsets) & 0xFFFF).to(tl.int32)
        m_rounded_bits = (m_bits + m_jitter) & -65536
        m_rounded = m_rounded_bits.to(tl.float32, bitcast=True).to(tl.bfloat16)

        v_bits = v.to(tl.int32, bitcast=True)
        v_jitter = (tl.randint(seed_v, offsets) & 0xFFFF).to(tl.int32)
        v_rounded_bits = (v_bits + v_jitter) & -65536
        v_rounded = v_rounded_bits.to(tl.float32, bitcast=True).to(tl.bfloat16)

        tl.store(m_buf_ptr + offsets, m_rounded, mask=mask)
        tl.store(v_buf_ptr + offsets, v_rounded, mask=mask)


def _triton_param_step(
    p, grad, m_buf, v_buf, comp, lr, beta1, beta2, eps, wd, step_t,
    base_seed=SEED, param_id=0, block_size=1024,
) -> None:
    n_elements = p.numel()
    if n_elements == 0:
        return

    p_c = p.contiguous().view(-1)
    g_c = grad.contiguous().view(-1)
    m_c = m_buf.contiguous().view(-1)
    v_c = v_buf.contiguous().view(-1)
    c_c = comp.contiguous().view(-1)

    bias_corr1 = 1.0 - (beta1 ** step_t)
    bias_corr2 = 1.0 - (beta2 ** step_t)
    # param_id is mixed in so different tensors don't share the same jitter
    seed_m = int((base_seed * 1_000_003 + step_t * 97 + param_id * 7919) & 0x7FFFFFFF)
    seed_v = int((base_seed * 1_000_033 + step_t * 193 + param_id * 104729) & 0x7FFFFFFF)

    grid = (triton.cdiv(n_elements, block_size),)
    _topo_adamw_fused_step_kernel[grid](
        p_c, g_c, m_c, v_c, c_c,
        n_elements,
        float(lr), float(beta1), float(beta2), float(eps), float(wd),
        float(bias_corr1), float(bias_corr2),
        seed_m, seed_v,
        P_IS_BF16=(p_c.dtype == torch.bfloat16),
        BLOCK_SIZE=block_size,
    )

    # .contiguous() returned copies for non-contiguous tensors; write back
    if not p.is_contiguous():
        p.copy_(p_c.view_as(p))
    if not m_buf.is_contiguous():
        m_buf.copy_(m_c.view_as(m_buf))
    if not v_buf.is_contiguous():
        v_buf.copy_(v_c.view_as(v_buf))
    if not comp.is_contiguous():
        comp.copy_(c_c.view_as(comp))


def _pytorch_param_step(p, m_buf, v_buf, comp, lr, beta1, beta2, eps, wd, step_t) -> None:
    g = p.grad.float()
    m = m_buf.float().mul_(beta1).add_(g, alpha=1.0 - beta1)
    v = v_buf.float().mul_(beta2).addcmul_(g, g, value=1.0 - beta2)
    del g

    # .clone() so fp32 params are not mutated in place by the arithmetic below
    w = p.detach().float().clone()
    denom = (v.sqrt() / math.sqrt(1.0 - beta2 ** step_t)).add_(eps)
    y = m.div(denom).mul_(-lr / (1.0 - beta1 ** step_t))
    del denom

    y.add_(w, alpha=-lr * wd)
    if p.dtype == torch.bfloat16:
        y.add_(comp.float())
    _stochastic_round_(m, m_buf)  # modifies m, which is not needed afterwards
    _stochastic_round_(v, v_buf)
    del m, v

    total = w.add_(y)
    if p.dtype == torch.bfloat16:
        new = total.to(torch.bfloat16)
        comp.copy_(total.sub_(new.float()))
        p.copy_(new)
    else:
        p.copy_(total.to(p.dtype))  # fp32 params keep full precision
        comp.zero_()
    del y, w, total


class TopoAdamW(Optimizer):
    """AdamW with bf16 moments (stochastic rounding) and compensated bf16 params.

    If `embedding_layer` is given AND its weight is one of the optimized
    parameters, the rows in `prime_anchors` are frozen to their initial values
    (gradient zeroed, value restored after each step). If the embedding is not
    being trained, anchoring is skipped and reported as inactive.
    """

    def __init__(
        self,
        params,
        lr=1e-4,
        betas=(0.9, 0.999),
        eps=1e-8,
        weight_decay=1e-2,
        embedding_layer=None,
        prime_anchors=None,
        base_seed=SEED,
        use_triton=True,
    ):
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay))
        self.base_seed = base_seed
        self.use_triton = bool(use_triton and HAS_TRITON)
        self.embedding_layer = embedding_layer

        self._param_ids = {}
        for g in self.param_groups:
            for p in g["params"]:
                self._param_ids[id(p)] = len(self._param_ids)

        self.anchors_active = False
        self.valid_anchors = []
        self.baseline = {}
        self.certified_sha256_digest = None

        if embedding_layer is not None:
            emb_w = embedding_layer.weight
            optimized = any(p is emb_w for g in self.param_groups for p in g["params"])
            if optimized:
                anchors = list(prime_anchors) if prime_anchors is not None else list(PRIME_ANCHORS)
                self.valid_anchors = [i for i in anchors if i < emb_w.shape[0]]
                self.baseline = {i: emb_w.data[i].detach().clone() for i in self.valid_anchors}
                self.certified_sha256_digest = self._digest()
                self.anchors_active = bool(self.valid_anchors)

        print(
            f"[TopoAdamW] Triton={self.use_triton} | anchors_active={self.anchors_active}"
            + (f" | digest {self.certified_sha256_digest[:16]}" if self.anchors_active else "")
        )

    def _digest(self) -> str:
        w = self.embedding_layer.weight.data
        buf = torch.cat([w[i].flatten().float() for i in self.valid_anchors]).cpu().numpy().tobytes()
        return hashlib.sha256(buf).hexdigest()

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        emb_w = self.embedding_layer.weight if self.anchors_active else None
        if emb_w is not None and emb_w.grad is not None:
            for i in self.valid_anchors:
                emb_w.grad[i].zero_()

        for group in self.param_groups:
            lr, wd, eps = group["lr"], group["weight_decay"], group["eps"]
            beta1, beta2 = group["betas"]
            for p in group["params"]:
                if not p.requires_grad or p.grad is None:
                    continue
                state = self.state[p]
                if not state:
                    state["step"] = 0
                    state["exp_avg"] = torch.zeros_like(p, dtype=torch.bfloat16)
                    state["exp_avg_sq"] = torch.zeros_like(p, dtype=torch.bfloat16)
                    state["comp"] = torch.zeros_like(p, dtype=torch.bfloat16)

                state["step"] += 1
                t = state["step"]
                m_buf, v_buf, comp = state["exp_avg"], state["exp_avg_sq"], state["comp"]
                grad_on_dev = p.grad.to(p.device) if p.grad.device != p.device else p.grad

                if self.use_triton and p.is_cuda and grad_on_dev.is_cuda:
                    _triton_param_step(
                        p, grad_on_dev, m_buf, v_buf, comp, lr, beta1, beta2, eps, wd, t,
                        base_seed=self.base_seed, param_id=self._param_ids[id(p)],
                    )
                else:
                    orig_grad = p.grad
                    p.grad = grad_on_dev
                    try:
                        _pytorch_param_step(p, m_buf, v_buf, comp, lr, beta1, beta2, eps, wd, t)
                    finally:
                        p.grad = orig_grad

                if emb_w is not None and p is emb_w:
                    for i in self.valid_anchors:
                        p[i].copy_(self.baseline[i].to(device=p.device, dtype=p.dtype))
                        comp[i].zero_()

        return loss

    def audit_telemetry(self):
        """Returns (max_drift, digest, matched), or (None, None, None) if anchoring is inactive."""
        if not self.anchors_active:
            return None, None, None
        w = self.embedding_layer.weight.data
        drift = max(
            torch.max(torch.abs(w[i].float() - self.baseline[i].to(w.device).float())).item()
            for i in self.valid_anchors
        )
        h = self._digest()
        return drift, h, h == self.certified_sha256_digest


def selftest_optimizer():
    """Triton vs PyTorch single-step equivalence (parameters + compensation only;
    moments differ because stochastic rounding uses different RNG streams)."""
    if not (HAS_TRITON and torch.cuda.is_available()):
        print("[selftest] skipped (needs Triton + CUDA)")
        return
    for dtype in (torch.bfloat16, torch.float32):
        torch.manual_seed(0)
        shape = (257, 129)
        p0 = torch.randn(shape, device="cuda").to(dtype)
        g = torch.randn(shape, device="cuda").to(torch.bfloat16)
        outs = []
        for use_t in (True, False):
            p = p0.clone()
            p.grad = g.clone()
            m = torch.zeros(shape, device="cuda", dtype=torch.bfloat16)
            v = torch.zeros_like(m)
            c = torch.zeros_like(m)
            if use_t:
                _triton_param_step(p, g, m, v, c, 1e-3, 0.9, 0.999, 1e-8, 1e-2, 1)
            else:
                _pytorch_param_step(p, m, v, c, 1e-3, 0.9, 0.999, 1e-8, 1e-2, 1)
            outs.append(p.float())
        diff = (outs[0] - outs[1]).abs().max().item()
        print(f"[selftest] {dtype}: max |triton - pytorch| = {diff:.3e}")


if os.environ.get("RUN_OPT_SELFTEST") == "1":
    selftest_optimizer()

# =====================================================================
# 2. MODEL / TOKENIZER (moonshotai/Kimi-VL-A3B-Instruct)
# =====================================================================
model_id = "moonshotai/Kimi-VL-A3B-Instruct"
max_seq_len = 512

SUPPORTED_ROPE_TYPES = {"linear", "dynamic", "yarn"}


def _normalize_rope_scaling(cfg):
    if cfg is None:
        return
    rs = getattr(cfg, "rope_scaling", None)
    if isinstance(rs, dict):
        scaling_type = rs.get("type") or rs.get("rope_type")
        if scaling_type not in SUPPORTED_ROPE_TYPES:
            cfg.rope_scaling = None
        else:
            rs["type"] = scaling_type
            if "factor" not in rs:
                rs["factor"] = 1.0


config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
_normalize_rope_scaling(config)
_normalize_rope_scaling(getattr(config, "text_config", None))

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True, trust_remote_code=True)
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

has_prequant = getattr(config, "quantization_config", None) is not None
model_kwargs = dict(
    config=config,
    attn_implementation="eager",
    device_map={"": 0},
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
if not has_prequant:
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        llm_int8_skip_modules=["lm_head"],  # keep lm_head in bf16 so it can be trained
    )

# The remote code defines tie_weights(self) without the `recompute_mapping`
# kwarg that newer transformers passes. Load the class directly and patch it.
KimiVLCls = get_class_from_dynamic_module(
    "modeling_kimi_vl.KimiVLForConditionalGeneration", model_id
)
_orig_tie_weights = KimiVLCls.tie_weights


def _tie_weights_compat(self, *args, recompute_mapping=None, **kwargs):
    return _orig_tie_weights(self, *args, **kwargs)


KimiVLCls.tie_weights = _tie_weights_compat

# The model card loads with `torch_dtype=` (it recommends transformers 4.48.2).
# Newer transformers (>= 4.56) renamed it to `dtype`; older ones don't know
# `dtype` and forward it to the model's __init__, which raises a TypeError.
import transformers
from packaging.version import Version

print(f"[*] transformers {transformers.__version__}")
_dtype_kw = "dtype" if Version(transformers.__version__) >= Version("4.56.0") else "torch_dtype"
try:
    model = KimiVLCls.from_pretrained(model_id, **{_dtype_kw: torch.bfloat16}, **model_kwargs)
except TypeError as e:
    if "unexpected keyword argument" not in str(e) or _dtype_kw not in str(e):
        raise
    _dtype_kw = "torch_dtype" if _dtype_kw == "dtype" else "dtype"
    print(f"[warn] retrying with {_dtype_kw}=")
    model = KimiVLCls.from_pretrained(model_id, **{_dtype_kw: torch.bfloat16}, **model_kwargs)
model.config.use_cache = False


# The remote MoEGate (DeepSeek-V3 style "noaux_tc" routing) contains
# `assert not self.training`, so it refuses to run in train mode. The routing
# itself (sigmoid scores -> top-k) is differentiable w.r.t. the gate output and
# needs no special training logic. Run the gate with training=False for the
# duration of its forward, and make sure the aux-loss slot is a tensor because
# DeepseekV3MoE's training branch unpacks (idx, weight, aux_loss).
def _make_train_safe_gate(gate):
    orig_forward = gate.forward

    def forward(hidden_states):
        was_training = gate.training
        gate.training = False
        try:
            out = orig_forward(hidden_states)
        finally:
            gate.training = was_training
        topk_idx, topk_weight, aux_loss = out
        if aux_loss is None:
            aux_loss = torch.zeros((), device=topk_idx.device, dtype=torch.float32)
        return topk_idx, topk_weight, aux_loss

    gate.forward = forward


_n_gates = 0
for _m in model.modules():
    if type(_m).__name__ == "MoEGate":
        _make_train_safe_gate(_m)
        _n_gates += 1
print(f"[*] Patched {_n_gates} MoE gates for training")
if _n_gates == 0:
    raise RuntimeError("No MoEGate modules found; the assert fix did not apply.")

gc.collect()
torch.cuda.empty_cache()

embedding_layer = model.get_input_embeddings()
if embedding_layer is None and hasattr(model, "language_model"):
    embedding_layer = model.language_model.get_input_embeddings()

# ---- choose trainable params: lm_head + language-model norms ----
for p in model.parameters():
    p.requires_grad = False

decay_params, no_decay_params, seen = [], [], set()
for name, mod in model.named_modules():
    is_head = name.endswith("lm_head")
    is_norm = name.endswith("norm") and "language_model" in name
    if not (is_head or is_norm):
        continue
    for p in mod.parameters(recurse=False):
        if id(p) in seen or not p.is_floating_point():
            continue  # skip quantized (uint8) tensors
        seen.add(id(p))
        p.requires_grad = True
        # no weight decay on norm scales / biases
        (decay_params if (is_head and p.ndim >= 2) else no_decay_params).append(p)

trainable_params = decay_params + no_decay_params
if not trainable_params:
    raise RuntimeError("No trainable parameters found; check module names.")

if hasattr(model, "gradient_checkpointing_enable"):
    try:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    except Exception:
        try:
            model.gradient_checkpointing_enable()
        except Exception:
            model.language_model.gradient_checkpointing_enable()

try:
    model.enable_input_require_grads()
except Exception as e:
    print(f"[warn] enable_input_require_grads failed: {e!r}")

print(f"[*] Trainable parameters: {sum(p.numel() for p in trainable_params):,}")

WEIGHT_DECAY = 1e-2
topo_optimizer = TopoAdamW(
    [
        {"params": decay_params, "weight_decay": WEIGHT_DECAY},
        {"params": no_decay_params, "weight_decay": 0.0},
    ],
    lr=1e-4,
    embedding_layer=embedding_layer,
    prime_anchors=PRIME_ANCHORS,
    base_seed=SEED,
    use_triton=True,
)


def _get_input_device() -> torch.device:
    emb = embedding_layer
    if emb is not None and hasattr(emb, "weight") and emb.weight.device.type != "meta":
        return emb.weight.device
    return primary_device


input_device = _get_input_device()
print(f"[*] Input target device: {input_device}")

# =====================================================================
# 3. DATA
# =====================================================================
# Use the token the chat template actually uses to end an assistant turn.
_im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
if _im_end_id is not None and _im_end_id != tokenizer.unk_token_id:
    END_TOKEN = "<|im_end|>"
else:
    END_TOKEN = tokenizer.eos_token or ""
print(f"[*] End-of-turn token used for labels: {END_TOKEN!r}")


def build_user_prompt(sample):
    instruction = (sample.get("instruction") or "").strip()
    extra_input = (sample.get("input") or "").strip()
    return f"{instruction}\n\nInput:\n{extra_input}" if extra_input else instruction


def preprocess_function(sample):
    user = build_user_prompt(sample)
    if getattr(tokenizer, "chat_template", None):
        try:
            prompt_text = tokenizer.apply_chat_template(
                [{"role": "user", "content": user}],
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        except TypeError:
            prompt_text = tokenizer.apply_chat_template(
                [{"role": "user", "content": user}],
                tokenize=False,
                add_generation_prompt=True,
            )
    else:
        bos = tokenizer.bos_token or ""
        prompt_text = f"{bos}{user}\n"

    answer_text = (sample.get("output") or "") + END_TOKEN
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(answer_text, add_special_tokens=False)["input_ids"]
    input_ids = (list(prompt_ids) + answer_ids)[:max_seq_len]
    labels = ([-100] * len(prompt_ids) + answer_ids)[:max_seq_len]
    return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids), "labels": labels}


def has_supervised_tokens(ex):
    # Drop samples whose prompt fills the window (all labels masked -> 0/0 NaN loss)
    return any(l != -100 for l in ex["labels"])


raw = load_dataset("yahma/alpaca-cleaned", split="train").shuffle(seed=SEED)
train_ds = (
    raw.select(range(12500))
    .map(preprocess_function, remove_columns=raw.column_names)
    .filter(has_supervised_tokens)
)
eval_ds = (
    raw.select(range(12500, 13000))
    .map(preprocess_function, remove_columns=raw.column_names)
    .filter(has_supervised_tokens)
)
print(f"[*] train samples: {len(train_ds)} | eval samples: {len(eval_ds)}")

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True
)
pin = use_cuda and input_device.type == "cuda"
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, collate_fn=collator, pin_memory=pin)
eval_loader = DataLoader(eval_ds, batch_size=2, shuffle=False, collate_fn=collator, pin_memory=pin)

# =====================================================================
# 4. TRAINING & AUDIT LOOP
# =====================================================================
epochs = 3
grad_accum = 16
max_grad_norm = 1.0
total_steps = (len(train_loader) * epochs) // grad_accum  # matches actual optimizer steps
scheduler = get_cosine_schedule_with_warmup(
    topo_optimizer, num_warmup_steps=20, num_training_steps=total_steps
)


@torch.no_grad()
def evaluate_loss():
    model.eval()
    losses = []
    for batch in eval_loader:
        batch = {k: v.to(input_device, non_blocking=pin) for k, v in batch.items()}
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            out = model(**batch)
        losses.append(out.loss.item())
    model.train()
    return sum(losses) / max(len(losses), 1)


def save_trainable(path):
    state = {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}
    torch.save(state, os.path.join(path, "trainable_weights.pt"))


print(f"[baseline] validation loss before training: {evaluate_loss():.4f}", flush=True)

# Mount Drive first in Colab:  from google.colab import drive; drive.mount("/content/drive")
output_dir = "/content/drive/MyDrive/topogpt/Kimi-VL-A3B-Instruct-alpaca-topo-4bit-triton-train"
os.makedirs(output_dir, exist_ok=True)

model.train()
topo_optimizer.zero_grad()
global_step = 0
micro_step = 0  # global across epochs, so leftover micro-batches complete a full window
running_step_loss = 0.0
accum_loss = 0.0

for epoch in range(1, epochs + 1):
    for batch in train_loader:
        batch = {k: v.to(input_device, non_blocking=pin) for k, v in batch.items()}
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / grad_accum

        loss.backward()
        accum_loss += outputs.loss.detach().item() / grad_accum
        micro_step += 1

        if micro_step % grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(trainable_params, max_grad_norm)
            topo_optimizer.step()
            scheduler.step()
            topo_optimizer.zero_grad()

            global_step += 1
            running_step_loss += accum_loss
            accum_loss = 0.0

            if global_step % 20 == 0:
                avg_loss = running_step_loss / 20.0
                lr_now = scheduler.get_last_lr()[0]
                print(f"step {global_step:>5} | train loss {avg_loss:.4f} | lr {lr_now:.2e}", flush=True)
                running_step_loss = 0.0

            if global_step % 100 == 0:
                print(f"[eval] step {global_step:>5} | val loss {evaluate_loss():.4f}", flush=True)

    epoch_dir = os.path.join(output_dir, f"epoch-{epoch}")
    os.makedirs(epoch_dir, exist_ok=True)
    save_trainable(epoch_dir)  # only the trained tensors, not the full 4-bit model
    print(f"[*] Saved epoch {epoch} trainable weights -> {epoch_dir}", flush=True)

max_drift, current_hash, hash_matched = topo_optimizer.audit_telemetry()
if max_drift is None:
    print("[TopoAudit] anchors inactive (embedding is frozen), nothing to audit.")
else:
    print(f"[TopoAudit] Max Anchor Drift: {max_drift:.6e} | Hash Matched: {hash_matched} | Digest: {current_hash[:16]}")

save_trainable(output_dir)
tokenizer.save_pretrained(output_dir)
print("[*] Done.")

config.json: 0.00B [00:00, ?B/s]

configuration_kimi_vl.py: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_moonshot.py: 0.00B [00:00, ?B/s]

tiktoken.model:   0%|          | 0.00/2.80M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

... (18 outputs left out)

Filter:   0%|          | 0/12500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

[*] train samples: 12500 | eval samples: 500
[baseline] validation loss before training: 1.0427
step    20 | train loss 1.0025 | lr 1.00e-04
step    40 | train loss 1.0167 | lr 9.99e-05
step    60 | train loss 1.0176 | lr 9.97e-05
step    80 | train loss 0.9793 | lr 9.93e-05
step   100 | train loss 0.9938 | lr 9.88e-05
[eval] step   100 | val loss 0.9953
step   120 | train loss 0.9648 | lr 9.81e-05
step   140 | train loss 0.9705 | lr 9.73e-05
step   160 | train loss 1.0146 | lr 9.64e-05
step   180 | train loss 1.0200 | lr 9.53e-05
step   200 | train loss 0.9653 | lr 9.41e-05
[eval] step   200 | val loss 0.9924
step   220 | train loss 0.9848 | lr 9.27e-05
step   240 | train loss 1.0024 | lr 9.13e-05
step   260 | train loss 0.9873 | lr 8.97e-05
step   280 | train loss 0.9929 | lr 8.79e-05
step   300 | train loss 0.9807 | lr 8.61e-05
[eval] step   300 | val loss 0.9915
step   320 | train loss 0.9545 | lr 8.42e-05
step   340 | train loss 0.9815 | lr 8.21e-05
step   360 | train loss 0.9774 

In [2]:
%cd /content/drive/MyDrive/topogpt/Kimi-VL-A3B-Instruct-alpaca-topo-4bit-triton-train
!du -sh *

/content/drive/MyDrive/topogpt/Kimi-VL-A3B-Instruct-alpaca-topo-4bit-triton-train
641M	epoch-1
641M	epoch-2
641M	epoch-3
1.0K	special_tokens_map.json
2.7M	tiktoken.model
4.0K	tokenizer_config.json
641M	trainable_weights.pt


## merger

In [3]:
import os, json, shutil, torch
from huggingface_hub import snapshot_download
from safetensors.torch import load_file, save_file

TRAINED = "/content/drive/MyDrive/topogpt/Kimi-VL-A3B-Instruct-alpaca-topo-4bit-triton-train/epoch-1/trainable_weights.pt"
OUT = "/content/merged"

src = snapshot_download("moonshotai/Kimi-VL-A3B-Instruct")
os.makedirs(OUT, exist_ok=True)
for f in os.listdir(src):  # config, tokenizer, index json, remote code, etc.
    if not f.endswith(".safetensors") and os.path.isfile(os.path.join(src, f)):
        shutil.copyfile(os.path.join(src, f), os.path.join(OUT, f))

trained = torch.load(TRAINED, map_location="cpu")
wmap = json.load(open(os.path.join(src, "model.safetensors.index.json")))["weight_map"]
assert all(k in wmap for k in trained), [k for k in trained if k not in wmap][:5]

replaced = changed = 0
for shard in sorted(set(wmap.values())):
    t = load_file(os.path.join(src, shard))
    for k in (k for k in trained if wmap[k] == shard):
        assert t[k].shape == trained[k].shape, k
        new = trained[k].to(t[k].dtype).contiguous()
        changed += int(not torch.equal(t[k], new))
        t[k] = new
        replaced += 1
    save_file(t, os.path.join(OUT, shard), metadata={"format": "pt"})
    print(shard, "done")

assert replaced == len(trained), (replaced, len(trained))
print(f"merged {replaced} tensors ({changed} differ from base) -> {OUT}")

Fetching 25 files:   0%|          | 0/25 [00:00<?, ?it/s]

figures/logo.png:   0%|          | 0.00/13.1k [00:00<?, ?B/s]

figures/instruct_perf.png:   0%|          | 0.00/2.24M [00:00<?, ?B/s]

figures/demo.png:   0%|          | 0.00/525k [00:00<?, ?B/s]

figures/arch.png:   0%|          | 0.00/641k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

image_processing_kimi_vl.py: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

processing_kimi_vl.py: 0.00B [00:00, ?B/s]

model-00001-of-00007.safetensors done
model-00002-of-00007.safetensors done
model-00003-of-00007.safetensors done
model-00004-of-00007.safetensors done
model-00005-of-00007.safetensors done
model-00006-of-00007.safetensors done
model-00007-of-00007.safetensors done
merged 83 tensors (83 differ from base) -> /content/merged


## gguf-convertor

In [4]:
import os, subprocess

LLAMA_CPP_DIR = "/content/drive/MyDrive/topogpt/llama.cpp"
MERGED    = "/content/merged"
GGUF_BF16 = "/content/kimi-vl-ft-bf16.gguf"

subprocess.run(["python", f"{LLAMA_CPP_DIR}/convert_hf_to_gguf.py", MERGED,
                "--outfile", GGUF_BF16, "--outtype", "bf16"], check=True)

print(GGUF_BF16, f"{os.path.getsize(GGUF_BF16)/1e9:.1f} GB")

/content/kimi-vl-ft-bf16.gguf 31.9 GB


In [5]:
!chmod a+x /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize
!ls -ltha /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize

-rwx------ 1 root root 18K Nov  2 10:56 /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize


## gguf-quantize

In [6]:
import os, subprocess

LLAMA_CPP_DIR = "/content/drive/MyDrive/topogpt/llama.cpp"
GGUF_BF16 = "/content/kimi-vl-ft-bf16.gguf"
GGUF_Q8   = "/content/drive/MyDrive/GGUF_Models/kimi-vl-ft-Q8_0.gguf"

BIN_DIR = f"{LLAMA_CPP_DIR}/build/bin"
BIN = f"{BIN_DIR}/llama-quantize"
env = {**os.environ, "LD_LIBRARY_PATH": f"{BIN_DIR}:{os.environ.get('LD_LIBRARY_PATH', '')}"}

if not os.path.exists(BIN):
    subprocess.run(f"cd {LLAMA_CPP_DIR} && cmake -B build -DLLAMA_CURL=OFF && cmake --build build -j --target llama-quantize",
                   shell=True, check=True)

subprocess.run([BIN, GGUF_BF16, GGUF_Q8, "Q8_0"], check=True, env=env)
print(GGUF_Q8, f"{os.path.getsize(GGUF_Q8)/1e9:.1f} GB")

/content/drive/MyDrive/GGUF_Models/kimi-vl-ft-Q8_0.gguf 17.0 GB


In [7]:
from llama_cpp import Llama

GGUF = "/content/drive/MyDrive/GGUF_Models/kimi-vl-ft-Q8_0.gguf"

llm = Llama(model_path=GGUF, n_ctx=2048, n_gpu_layers=-1, verbose=False)

out = llm.create_chat_completion(
    messages=[{"role": "user", "content": "Explain what a mixture-of-experts model is in two sentences."}],
    max_tokens=128,
    temperature=0.2,  # the model card recommends 0.2 for Instruct models
)
print(out["choices"][0]["message"]["content"])

A mixture-of-experts model is a type of ensemble learning method that combines the predictions of multiple experts, each trained on a different subset of the data or using a different model. The final prediction is made by weighting the predictions of the experts according to their performance.


## hf

In [9]:
import os
from google.colab import userdata
from huggingface_hub import HfApi, login

HF_TOKEN = userdata.get('HF_TOKEN')
user_or_name = 'frankmorales2020'

REPO_ID = f"{user_or_name}/Kimi-VL-A3B-Instruct-alpaca-ft-GGUF"
GGUF = "/content/drive/MyDrive/GGUF_Models/kimi-vl-ft-Q8_0.gguf"

login(token=HF_TOKEN)
api = HfApi()
api.create_repo(REPO_ID, repo_type="model", private=False, exist_ok=True)

README = """---
license: mit
base_model: moonshotai/Kimi-VL-A3B-Instruct
tags: [gguf, llama.cpp, kimi-vl, fine-tune]
---
# Kimi-VL-A3B-Instruct, Alpaca fine-tune (GGUF, Q8_0)

Text-only GGUF of moonshotai/Kimi-VL-A3B-Instruct. Only `lm_head` and the language-model
norm layers were fine-tuned (about 335M parameters, 4-bit base, yahma/alpaca-cleaned,
12,500 examples, 1 epoch checkpoint). Validation loss improved from 1.043 to 0.990.
No vision projector (mmproj) is included.
"""
api.upload_file(path_or_fileobj=README.encode(), path_in_repo="README.md", repo_id=REPO_ID)
api.upload_file(path_or_fileobj=GGUF, path_in_repo=os.path.basename(GGUF), repo_id=REPO_ID)
print(f"https://huggingface.co/{REPO_ID}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...dels/kimi-vl-ft-Q8_0.gguf:   1%|1         |  216MB / 17.0GB            

https://huggingface.co/frankmorales2020/Kimi-VL-A3B-Instruct-alpaca-ft-GGUF


## inference-hf

In [1]:
from google.colab import userdata
from huggingface_hub import login
from llama_cpp import Llama

login(token=userdata.get('HF_TOKEN'))  # needed while the repo is private

llm = Llama.from_pretrained(
    repo_id="frankmorales2020/Kimi-VL-A3B-Instruct-alpaca-ft-GGUF",
    filename="kimi-vl-ft-Q8_0.gguf",
    n_ctx=2048,
    n_gpu_layers=-1,
    verbose=False,
)

prompts = [
    "Explain what a mixture-of-experts model is in two sentences.",
    "Give three tips for writing a clear email.",
    "Rewrite this sentence more formally: 'hey, can u send me the report asap?'",
]

for p in prompts:
    out = llm.create_chat_completion(
        messages=[{"role": "user", "content": p}],
        max_tokens=200,
        temperature=0.2,  # the model card recommends 0.2 for Instruct models
    )
    print(f"### {p}\n{out['choices'][0]['message']['content']}\n")

./kimi-vl-ft-Q8_0.gguf:   0%|          | 0.00/17.0G [00:00<?, ?B/s]

### Explain what a mixture-of-experts model is in two sentences.
A mixture-of-experts model is a type of ensemble learning method that combines the predictions of multiple experts, each trained on a different subset of the data or using a different model. The final prediction is made by weighting the predictions of the experts according to their performance.

### Give three tips for writing a clear email.
1. Be concise and to the point: Keep your email brief and focused on the main topic. Avoid using long paragraphs or unnecessary details.

2. Use a clear and descriptive subject line: Your subject line should accurately reflect the content of your email and grab the recipient's attention.

3. Use proper grammar and spelling: Proofread your email before sending it to ensure that it is free of errors and easy to read. This will help to convey your message more effectively.

### Rewrite this sentence more formally: 'hey, can u send me the report asap?'
Greetings, could you please send me 

In [2]:
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv

utilization.gpu [%], memory.used [MiB]
0 %, 16915 MiB
